# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The purpose of this queue is to prioritize pages for **human review**, not to automatically decide which pages should be refreshed. The ranking combines the model's validated decline signal with observable SEO performance signals available in the evaluation data.

Each queued page receives one or more reason codes explaining why it was prioritized. These codes describe observed signals in the data; they should not be interpreted as causal explanations for a page's performance.

The queue is therefore a **decision-support artifact**: pages near the top are candidates for earlier review, while the final action remains with a human reviewer.


In [1]:
from google.colab import userdata
HF_TOKEN = "code"

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# 1-15 days
features = con.sql(f"""
SELECT client_hash_id, content_hash_id,
SUM(gsc_impressions) as imp_first15,
SUM(gsc_clicks) as clk_first15,
AVG(gsc_sum_position / NULLIF(gsc_impressions,0)) AS avg_position_first15,
SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions),0) AS ctr_first15,
MAX(client_has_ga4::INT) AS has_ga4
FROM {FACT_MAR}
WHERE gsc_data_available IS TRUE
AND CAST(SUBSTR(CAST(report_date AS VARCHAR),9,2) AS INT) <= 15
GROUP BY 1,2
HAVING imp_first15 >= 50

""").df()

# 16-31
label_data = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clk_last15
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND CAST(SUBSTR(CAST(report_date AS VARCHAR), 9, 2) AS INT) > 15
    GROUP BY 1,2
""").df()

data = features.merge(label_data, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining'] = (data['clk_last15'] < data['clk_first15']).astype(int)

from sklearn.model_selection import GroupShuffleSplit

feature_cols = ['imp_first15','clk_first15','avg_position_first15','ctr_first15','has_ga4']
X = data[feature_cols].replace([np.inf,-np.inf],np.nan).fillna(0)
y = data['is_declining']
groups = data['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size = 0.2 , random_state = 42)
train_idx , test_idx = next(gss.split(X,y, groups))
X_train , X_test = X.iloc[train_idx] , X.iloc[test_idx]
y_train , y_test = y.iloc[train_idx] , y.iloc[test_idx]
data_test = data.iloc[test_idx].copy()
#confirm no clients appears in both sets
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:,1]



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,247 rows | declining rate: 0.289


In [4]:
X_test['scores'] = model_scores

X_test.sort_values(by='scores',ascending=False).head()


/tmp/ipykernel_667/2416206201.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test['scores'] = model_scores


,imp_first15,clk_first15,avg_position_first15,ctr_first15,has_ga4,scores
40076,87.0,1.0,64.540193,0.011494,1,1.00
86210,288.0,3.0,56.023446,0.010417,1,0.99
86872,135.0,2.0,7.616541,0.014815,1,0.99
40969,59.0,1.0,4.766667,0.016949,1,0.99
40104,142.0,1.0,58.888040,0.007042,1,0.99


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use**

Decision-support priority scoring to guide content editorial teams toward high-ROI refresh candidates.

**Known Limits**

Non-Causal: Model ranks observed traffic decay opportunity; it does not guarantee search engine rank restoration.
Snapshot Data: Evaluation is based on a 90-day static window snapshot; seasonal traffic variations require human context.
Unranked Proxy: Pages with avg_position = 0 are assigned a default proxy position of 50.0.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Rules**

Editors must read the target URL before making updates to confirm that current search intent matches the refresh topic.
Verify that traffic decline is not caused by permanent brand query shifts or product deprecation.

**No-Go List (What NOT to Automate)**

🚫 Auto-Publishing AI Content: Never automatically publish AI-generated text directly to live CMS without human editor review.

🚫 Automated 301 Redirects: Never auto-redirect or delete URLs based solely on low model priority scores.

🚫 Automated Site-wide Title Changes: Never run bulk automated meta updates across entire client domains without human verification.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring:**
 Track post-refresh CTR and click recovery over a 30-day post-update window.

**Retrain Triggers:**


1.   Precision Drop: Retrain model if Precision@50 on weekly holdout data drops below 85%.
2.   Domain Distribution Shift: Retrain model when new client domain profiles (e.g. e-commerce vs SaaS) are onboarded.





In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.